# EP1 - Agente RAG para Hotel El Almendro

## Ingeniería de Soluciones con Inteligencia Artificial

Este cuaderno implementa un prototipo RAG para apoyar al área de Obras, Mantención y Mejoras (OMM) de Hotel El Almendro.

El flujo utilizado es:

1. Cargar documentos PDF.
2. Dividir los documentos en chunks.
3. Generar embeddings.
4. Crear una base vectorial con FAISS.
5. Recuperar los fragmentos más relevantes.
6. Enviar el contexto recuperado al modelo de lenguaje.
7. Generar una respuesta basada únicamente en la documentación.

### Modelos utilizados

- `mistral-embed`: generación de embeddings.
- `mistral-small-latest`: generación de respuestas.


# Instalación de dependencias


In [23]:
# --- Instalación de dependencias ---

%pip install -qU langchain-openai langchain-text-splitters faiss-cpu pypdf

# Configuración de credenciales

En Google Colab se debe crear el Secret `LLM_API_KEY` con la API key de Mistral.


In [21]:
# --- Configuración de credenciales ---

import os
from google.colab import userdata

LLM_API_KEY = userdata.get("LLM_API_KEY")

if not LLM_API_KEY:
    raise ValueError(
        "No se encontró el Secret LLM_API_KEY. "
        "Agrega tu API key de Mistral en los Secrets de Google Colab."
    )

LLM_MODEL = "ministral-8b-2512"
EMBEDDING_MODEL = "mistral-embed"

os.environ["LLM_API_KEY"] = LLM_API_KEY
os.environ["LLM_BASE_URL"] = "https://api.mistral.ai/v1"
os.environ["LLM_MODEL"] = LLM_MODEL
os.environ["EMBEDDING_MODEL"] = EMBEDDING_MODEL

print("Credenciales y variables de entorno configuradas correctamente.")
print(f"Modelo de generación: {LLM_MODEL}")
print(f"Modelo de embeddings: {EMBEDDING_MODEL}")

Credenciales y variables de entorno configuradas correctamente.
Modelo de generación: ministral-8b-2512
Modelo de embeddings: mistral-embed


# Configuración de LangChain y Mistral


In [16]:
from langchain_openai import ChatOpenAI
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.messages import HumanMessage

from pypdf import PdfReader

import faiss
import numpy as np
import requests
import os


llm = ChatOpenAI(
    base_url=os.getenv("LLM_BASE_URL"),
    api_key=os.getenv("LLM_API_KEY"),
    model=os.getenv("LLM_MODEL"),
    temperature=0.1,
    max_tokens=500
)


class MistralEmbeddings:
    def __init__(self, api_key, model="mistral-embed"):
        self.api_key = api_key
        self.model = model
        self.url = "https://api.mistral.ai/v1/embeddings"

    def embed_documents(self, textos):
        headers = {
            "Authorization": f"Bearer {self.api_key}",
            "Content-Type": "application/json"
        }

        payload = {
            "model": self.model,
            "input": textos
        }

        response = requests.post(
            self.url,
            headers=headers,
            json=payload
        )

        response.raise_for_status()

        datos = response.json()["data"]

        return [item["embedding"] for item in datos]

    def embed_query(self, texto):
        return self.embed_documents([texto])[0]


embeddings = MistralEmbeddings(
    api_key=os.getenv("LLM_API_KEY"),
    model=os.getenv("EMBEDDING_MODEL")
)


print("Modelo de chat y modelo de embeddings configurados correctamente.")
print(f"Modelo de chat: {os.getenv('LLM_MODEL')}")
print(f"Modelo de embeddings: {os.getenv('EMBEDDING_MODEL')}")

Modelo de chat y modelo de embeddings configurados correctamente.
Modelo de chat: ministral-8b-2512
Modelo de embeddings: mistral-embed


## 1. Carga de documentos PDF



In [4]:
from google.colab import files

uploaded = files.upload()

pdf_files = [name for name in uploaded.keys() if name.lower().endswith(".pdf")]

print(f"Archivos PDF cargados: {len(pdf_files)}")

for pdf in pdf_files:
    print(f"- {pdf}")

Saving Manual-Caldera-No-Condensacion-Chile.pdf to Manual-Caldera-No-Condensacion-Chile.pdf
Archivos PDF cargados: 1
- Manual-Caldera-No-Condensacion-Chile.pdf


In [5]:
# Cargar el contenido de los PDF y conservar sus metadatos

documents = []

for pdf_file in pdf_files:
    reader = PdfReader(pdf_file)

    for page_number, page in enumerate(reader.pages, start=1):
        text = page.extract_text()

        if text and text.strip():
            documents.append({
                "texto": text.strip(),
                "documento": os.path.basename(pdf_file),
                "pagina": page_number
            })

print(f"Total de páginas con contenido: {len(documents)}")

for i, document in enumerate(documents[:3], start=1):
    print(f"Documento {i}: {document['documento']}")
    print(f"Página: {document['pagina']}")
    print(document["texto"][:300])
    print("-" * 60)

Total de páginas con contenido: 32
Documento 1: Manual-Caldera-No-Condensacion-Chile.pdf
Página: 1
3#3.&$-& 3&7
#3PD&JBOOJ3
3JOOBJ &DP3# 
3JOOBJ &DP3# 
Caldera de Alta Eficiencia
,
------------------------------------------------------------
Documento 2: Manual-Caldera-No-Condensacion-Chile.pdf
Página: 3
Derechos del Usuario
An t es de hac er  funcionar  la c alder a, lea las instruc ciones de usuario .
U n usuario tiene der echo a r ecibir  asist encia del servicio t écnic o aut oriz ado dur an t e el período de gar an tía a partir  de la f echa de c ompr a. 
N o obst an t e; el clien t e deber á p
------------------------------------------------------------
Documento 3: Manual-Caldera-No-Condensacion-Chile.pdf
Página: 4
Peligro
INFORMACIÓN DE SEGURIDAD
------------------------------------------------------------


## 2. Chunking con RecursiveCharacterTextSplitter

Se utiliza una configuración inicial de `chunk_size=350` y `chunk_overlap=50`.

In [6]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=350,
    chunk_overlap=50,
    length_function=len
)

chunks = []

for document in documents:
    text_chunks = text_splitter.split_text(document["texto"])

    for chunk_text in text_chunks:
        chunks.append({
            "texto": chunk_text,
            "documento": document["documento"],
            "pagina": document["pagina"]
        })

print(f"Cantidad total de chunks: {len(chunks)}")

for i, chunk in enumerate(chunks[:5], start=1):
    print(f"CHUNK {i}")
    print(f"Documento: {chunk['documento']}")
    print(f"Página: {chunk['pagina']}")
    print(chunk["texto"])
    print("-" * 60)

Cantidad total de chunks: 52
CHUNK 1
Documento: Manual-Caldera-No-Condensacion-Chile.pdf
Página: 1
3#3.&$-& 3&7
#3PD&JBOOJ3
3JOOBJ &DP3# 
3JOOBJ &DP3# 
Caldera de Alta Eficiencia
,
------------------------------------------------------------
CHUNK 2
Documento: Manual-Caldera-No-Condensacion-Chile.pdf
Página: 3
Derechos del Usuario
An t es de hac er  funcionar  la c alder a, lea las instruc ciones de usuario .
U n usuario tiene der echo a r ecibir  asist encia del servicio t écnic o aut oriz ado dur an t e el período de gar an tía a partir  de la f echa de c ompr a.
------------------------------------------------------------
CHUNK 3
Documento: Manual-Caldera-No-Condensacion-Chile.pdf
Página: 3
N o obst an t e; el clien t e deber á pagar  por  el Servicio T écnic o por  el mal funcionamien t o del pr oduct o c ausados por  la negligencia de clien t es 
y  / o de fuerz a may or .
Responsabilidades para Servicios Técnicos
(E l U suario es r esponsable por

## 3. Generación de embeddings y base vectorial FAISS

Cada chunk es transformado a una representación vectorial mediante `mistral-embed`. Luego los vectores se almacenan en FAISS para realizar búsquedas por similitud semántica.


In [7]:
try:
    textos = [chunk["texto"] for chunk in chunks]

    vectores = embeddings.embed_documents(textos)

    vectores_np = np.array(vectores, dtype="float32")

    faiss.normalize_L2(vectores_np)

    dimension = vectores_np.shape[1]

    vector_db = faiss.IndexFlatIP(dimension)
    vector_db.add(vectores_np)

    print("Base de datos vectorial FAISS creada correctamente.")
    print(f"Vectores almacenados: {vector_db.ntotal}")
    print(f"Dimensiones de cada embedding: {dimension}")

except Exception as e:
    print(f"Error al crear la base vectorial: {e}")

Base de datos vectorial FAISS creada correctamente.
Vectores almacenados: 52
Dimensiones de cada embedding: 1024


## 4. Recuperación de información

El retriever recupera inicialmente los 3 chunks con mayor similitud semántica respecto de la consulta.


In [8]:
TOP_K = 3

def recuperar_chunks(pregunta, k=TOP_K):
    vector_pregunta = embeddings.embed_query(pregunta)

    vector_pregunta_np = np.array(
        [vector_pregunta],
        dtype="float32"
    )

    faiss.normalize_L2(vector_pregunta_np)

    similitudes, indices = vector_db.search(
        vector_pregunta_np,
        k
    )

    resultados = []

    for posicion, indice in enumerate(indices[0]):
        if indice == -1:
            continue

        chunk = chunks[indice].copy()
        chunk["similitud"] = float(similitudes[0][posicion])

        resultados.append(chunk)

    return resultados


def recuperar_contexto(pregunta):
    relevant_chunks = recuperar_chunks(pregunta)

    print(f"Consulta: {pregunta}")
    print(f"Chunks recuperados: {len(relevant_chunks)}")
    print("-" * 60)

    for i, chunk in enumerate(relevant_chunks, start=1):
        print(f"CHUNK RELEVANTE {i}")
        print(f"Documento: {chunk['documento']}")
        print(f"Página: {chunk['pagina']}")
        print(f"Similitud: {chunk['similitud']:.4f}")
        print(chunk["texto"])
        print("-" * 60)

    return relevant_chunks

## 5. Prompt del agente OMM

El prompt establece el rol del agente, sus restricciones y el formato esperado de la respuesta.


In [9]:
prompt_template = """Eres un asistente técnico del área de Obras, Mantención y Mejoras (OMM) de Hotel El Almendro.

Responde únicamente utilizando la información incluida en el CONTEXTO recuperado.
No utilices conocimiento general para completar información que no aparezca en los documentos.
No inventes procedimientos, valores, frecuencias, repuestos, riesgos ni especificaciones técnicas.

Si el contexto no contiene información suficiente para responder, indica:
"No existe información suficiente en la documentación disponible".

Si existe información suficiente, responde de forma breve y clara e identifica el documento utilizado como fuente.
Si existen datos contradictorios entre documentos, informa la contradicción y no selecciones una respuesta sin evidencia de vigencia.

CONTEXTO:
{context}

PREGUNTA:
{question}

FORMATO DE RESPUESTA:
Respuesta:
Fuente:
Estado de respaldo: suficiente / insuficiente
"""

print("Prompt del agente configurado.")


Prompt del agente configurado.


## 6. Generación de respuesta con RAG


In [10]:
import time

def consultar_omm(pregunta):
    relevant_chunks = recuperar_chunks(pregunta)

    if not relevant_chunks:
        print("No se recuperaron fragmentos relevantes.")
        return None

    context_parts = []

    for chunk in relevant_chunks:
        context_parts.append(
            f"Documento: {chunk['documento']}\n"
            f"Página: {chunk['pagina']}\n"
            f"Contenido: {chunk['texto']}"
        )

    context = "\n\n".join(context_parts)

    prompt = prompt_template.format(
        context=context,
        question=pregunta
    )

    for intento in range(3):
        try:
            response = llm.invoke([
                HumanMessage(content=prompt)
            ])

            print("CONSULTA")
            print(pregunta)

            print("\nRESPUESTA")
            print(response.content)

            print("\nFRAGMENTOS UTILIZADOS")

            for i, chunk in enumerate(relevant_chunks, start=1):
                print(
                    f"{i}. {chunk['documento']} - "
                    f"Página {chunk['pagina']} - "
                    f"Similitud {chunk['similitud']:.4f}"
                )

            return {
                "pregunta": pregunta,
                "respuesta": response.content,
                "chunks": relevant_chunks
            }

        except Exception as e:
            if "429" in str(e) or "Rate limit" in str(e):
                if intento < 2:
                    print("Límite de solicitudes alcanzado. Esperando 30 segundos...")
                    time.sleep(30)
                else:
                    print("No fue posible generar la respuesta por límite temporal de la API.")
                    return None
            else:
                print(f"Error al generar la respuesta: {e}")
                return None

## 7. Prueba del sistema

Modifica la pregunta según el contenido de los documentos cargados.


In [17]:
# 7. Pregunta con información existente
pregunta_prueba = "¿Qué se debe hacer antes de realizar la limpieza o mantenimiento de la caldera?"
resultado = consultar_omm(pregunta_prueba)

CONSULTA
¿Qué se debe hacer antes de realizar la limpieza o mantenimiento de la caldera?

RESPUESTA
**Respuesta:**
Antes de realizar la limpieza o mantenimiento de la caldera, **debe desenchufarla**.

**Fuente:**
Manual-Caldera-No-Condensacion-Chile.pdf (Página 12).

**Estado de respaldo:** suficiente.

FRAGMENTOS UTILIZADOS
1. Manual-Caldera-No-Condensacion-Chile.pdf - Página 12 - Similitud 0.9089
2. Manual-Caldera-No-Condensacion-Chile.pdf - Página 5 - Similitud 0.8564
3. Manual-Caldera-No-Condensacion-Chile.pdf - Página 6 - Similitud 0.8360


## 8. Prueba de información inexistente

Esta prueba permite comprobar si el modelo respeta la restricción de no completar la respuesta utilizando conocimiento general.


In [19]:
# 8. Pregunta con información inexistente
pregunta_sin_contexto = "¿Cuál es el precio actual de una caldera nueva para el hotel?"
resultado_sin_contexto = consultar_omm(pregunta_sin_contexto)

CONSULTA
¿Cuál es el precio actual de una caldera nueva para el hotel?

RESPUESTA
Respuesta:
No existe información suficiente en la documentación disponible para determinar el precio actual de una caldera nueva.

Fuente:
- Manual-Caldera-No-Condensacion-Chile.pdf (no menciona precios ni costos).

Estado de respaldo: insuficiente.

FRAGMENTOS UTILIZADOS
1. Manual-Caldera-No-Condensacion-Chile.pdf - Página 7 - Similitud 0.8161
2. Manual-Caldera-No-Condensacion-Chile.pdf - Página 6 - Similitud 0.8139
3. Manual-Caldera-No-Condensacion-Chile.pdf - Página 12 - Similitud 0.8040


## 9. Prueba de recuperación semántica

Esta celda permite revisar directamente qué fragmentos recupera FAISS antes de generar la respuesta.


In [20]:
# 9. Recuperación semántica
pregunta_recuperacion = "¿Qué se debe hacer para limpiar el filtro de calefacción?"
chunks_recuperados = recuperar_contexto(pregunta_recuperacion)

Consulta: ¿Qué se debe hacer para limpiar el filtro de calefacción?
Chunks recuperados: 3
------------------------------------------------------------
CHUNK RELEVANTE 1
Documento: Manual-Caldera-No-Condensacion-Chile.pdf
Página: 6
Similitud: 0.8370
gama de voltaje.
Además, no trate de extender el cable de alimentación que tiene por defecto la caldera. 
No bloquee la rejilla de ventilación
Asegúrese de que cada válvula de los cuartos o habitaciones estén abiertas y que el aire se 
elimine correctamente
------------------------------------------------------------
CHUNK RELEVANTE 2
Documento: Manual-Caldera-No-Condensacion-Chile.pdf
Página: 12
Similitud: 0.8361
LIMPIEZA Y MANTENIMIENTO
* Mantenga la limpieza de la caldera en todo momento.
* Desenchufe la caldera antes de realizar la limpieza y / o mantenimiento.
* No limpie la caldera y mandos a distancia con un paño húmedo.
caldera.
* Después de la limpieza y / o mantenimiento, compruebe que todos los
componentes de la caldera están inta

## 10. Observaciones de las pruebas realizadas

Durante las pruebas del prototipo se verificó el funcionamiento del flujo RAG utilizando un manual técnico de caldera en formato PDF.

Se registraron los siguientes elementos:

- Pregunta realizada.
- Documento y página recuperada.
- Chunks recuperados mediante FAISS.
- Nivel de similitud de los fragmentos recuperados.
- Respuesta generada por el modelo.
- Fuente utilizada para respaldar la respuesta.
- Estado de respaldo de la respuesta.
- Comportamiento del sistema cuando la información no se encuentra disponible.

Las pruebas permitieron comprobar que el sistema puede recuperar información semánticamente relacionada desde el documento técnico y generar respuestas utilizando el contexto recuperado. También se verificó el comportamiento del agente ante una consulta cuya respuesta no se encontraba disponible en la documentación.